In [1]:
import warnings
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore")

PATH_C2L_INFECTED = "/Users/yashkulkarni/cellxgene_data/cell2location_infected_030525_yk.h5ad"
PATH_VISIUM_INFECTED = [
    "/Volumes/YK_Robey/spleen_visium/Spleen_Visium_Exp3A_V1S1_3wk_infected/outs/filtered_feature_bc_matrix.h5",
    "/Volumes/YK_Robey/spleen_visium/Spleen_Visium_Exp3A_V1S2_3wk_infected/outs/filtered_feature_bc_matrix.h5",
]

CLUSTER_KEY = "paper_clusters"
MZ_CLUSTER = "WP-C: MZ"
OUTDIR = Path("mz_marker_genes_0726_out")
OUTDIR.mkdir(exist_ok=True)

CURATED_MZ = ["Marco", "Gm2a", "Irf1", "Ltc4s", "Ccl4", "Col23a1", "Ccl24", "Igfbp2", "Vsig10"]

FDR = 0.05
MIN_LOG2FC = 0.5
MIN_PCT_IN = 0.25   # expressed in >=25% of MZ spots
TOP_N = 50


def load_matched_visium(c2l_path, visium_paths):
    c2l = sc.read_h5ad(c2l_path)
    samples = [sc.read_10x_h5(p) for p in visium_paths]
    for s in samples:
        s.var_names_make_unique()

    parsed = [(bc, bc.split("_")[0], "_".join(bc.split("_")[1:])) for bc in c2l.obs_names]
    obs_sets = [set(s.obs_names) for s in samples]
    suffix_to_sample = {
        suf: samples[
            max(range(len(samples)), key=lambda i: len({b for _, b, s in parsed if s == suf} & obs_sets[i]))
        ]
        for suf in {s for _, _, s in parsed}
    }

    matched = [bc for bc, base, suf in parsed if base in suffix_to_sample.get(suf, sc.AnnData()).obs_names]
    if not matched:
        raise ValueError(f"No matched spots for {c2l_path}")

    groups = OrderedDict()
    for bc in matched:
        base, suf = bc.split("_")[0], "_".join(bc.split("_")[1:])
        sample = suffix_to_sample[suf]
        groups.setdefault(id(sample), {"sample": sample, "pairs": []})["pairs"].append((base, bc))

    parts = []
    for g in groups.values():
        bases, c2l_bcs = zip(*g["pairs"])
        part = g["sample"][list(bases)].copy()
        part.obs_names = pd.Index(c2l_bcs)
        parts.append(part)

    out = sc.concat(parts, join="outer", fill_value=0)[matched].copy()
    out.obs[CLUSTER_KEY] = c2l[matched].obs[CLUSTER_KEY].values
    out.obs[CLUSTER_KEY] = out.obs[CLUSTER_KEY].astype("category")
    return out


adata = load_matched_visium(PATH_C2L_INFECTED, PATH_VISIUM_INFECTED)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print(adata.shape)
print(adata.obs[CLUSTER_KEY].value_counts())
print("MZ spots:", int((adata.obs[CLUSTER_KEY] == MZ_CLUSTER).sum()))

(1268, 32285)
paper_clusters
WP-C: MZ      242
RP-E: Rhag    222
WP-B: BZ      219
RP-A: GZMK    194
RP-D: F480    119
RP-C: NGP      90
RP-B: H_MK     75
WP-A: TZ       74
WP-D: GC       33
Name: count, dtype: int64
MZ spots: 242


In [2]:
# Binary label for one-vs-rest
adata.obs["is_MZ"] = np.where(adata.obs[CLUSTER_KEY] == MZ_CLUSTER, "MZ", "rest")
adata.obs["is_MZ"] = adata.obs["is_MZ"].astype("category")

sc.tl.rank_genes_groups(
    adata,
    groupby="is_MZ",
    groups=["MZ"],
    reference="rest",
    method="wilcoxon",
    pts=True,
    use_raw=False,
)

df = sc.get.rank_genes_groups_df(adata, group="MZ")
df = df.rename(columns={"names": "gene", "logfoldchanges": "log2FC", "pvals_adj": "padj"})

# Keep upregulated, significant, reasonably MZ-restricted
markers = df[
    (df["padj"] < FDR)
    & (df["log2FC"] >= MIN_LOG2FC)
    & (df["pct_nz_group"] >= MIN_PCT_IN)
].sort_values(["padj", "log2FC"], ascending=[True, False]).copy()

markers["rank"] = np.arange(1, len(markers) + 1)
markers.to_csv(OUTDIR / "MZ_vs_rest_infected_all_passing.csv", index=False)
df.to_csv(OUTDIR / "MZ_vs_rest_infected_full.csv", index=False)

print(f"Passing markers (padj<{FDR}, log2FC>={MIN_LOG2FC}, pct_MZ>={MIN_PCT_IN}): {len(markers)}")
display(markers.head(TOP_N))

# Where do the curated MZ genes land?
curated = df[df["gene"].isin(CURATED_MZ)].copy()
curated["passes_filters"] = (
    (curated["padj"] < FDR)
    & (curated["log2FC"] >= MIN_LOG2FC)
    & (curated["pct_nz_group"] >= MIN_PCT_IN)
)
curated = curated.sort_values("padj")
curated.to_csv(OUTDIR / "MZ_curated_genes_infected_stats.csv", index=False)
display(curated)

Passing markers (padj<0.05, log2FC>=0.5, pct_MZ>=0.25): 93


,gene,scores,log2FC,pvals,padj,pct_nz_group,pct_nz_reference,rank
11,Klf2,8.255033,0.517825,1.518602e-16,3.502005e-13,1.000000,0.976608,1
12,S100a4,8.196486,0.691463,2.475146e-16,5.063740e-13,0.942149,0.855750,2
13,Ltb,8.194828,0.562738,2.509520e-16,5.063740e-13,1.000000,0.973684,3
20,Ackr3,7.700209,0.725947,1.358439e-14,1.624341e-11,0.896694,0.783626,4
22,H2-DMb2,7.601461,0.515422,2.928065e-14,2.954143e-11,1.000000,0.985380,5
30,Slc35c2,7.093278,0.561758,1.309714e-12,8.996620e-10,0.904959,0.812865,6
33,Ccnd1,6.904564,0.621300,5.035775e-12,2.956000e-09,0.888430,0.784600,7
34,Ffar2,6.900076,0.742154,5.197483e-12,2.971841e-09,0.780992,0.607212,8
37,Stab2,6.893831,0.537389,5.430962e-12,2.971841e-09,0.962810,0.908382,9
41,Lfng,6.623737,0.503770,3.502296e-11,1.548926e-08,0.958678,0.913255,10


,gene,scores,log2FC,pvals,padj,pct_nz_group,pct_nz_reference,passes_filters
5,Gm2a,8.948905,0.430633,3.590270e-19,1.655884e-15,1.000000,0.988304,False
10,Irf1,8.262547,0.469166,1.425972e-16,3.502005e-13,1.000000,0.974659,False
83,Marco,5.991046,0.612389,2.084952e-09,4.986125e-07,0.826446,0.716374,True
188,Ltc4s,5.004053,1.087137,5.613740e-07,6.122959e-05,0.438017,0.263158,True
896,Ccl24,3.042068,0.658632,2.349592e-03,6.029935e-02,0.396694,0.286550,False
1216,Igfbp2,2.624437,0.783122,8.679235e-03,1.645385e-01,0.297521,0.198830,False
1388,Ccl4,2.456214,0.790890,1.404096e-02,2.342125e-01,0.280992,0.192982,False
1423,Col23a1,2.419232,0.685436,1.555331e-02,2.533495e-01,0.305785,0.226121,False
1499,Vsig10,2.350538,0.642981,1.874631e-02,2.904148e-01,0.322314,0.243665,False


In [3]:
from sklearn.neighbors import NearestNeighbors

# --- pull spatial from c2l onto matched adata ---
c2l = sc.read_h5ad(PATH_C2L_INFECTED)
assert "spatial" in c2l.obsm, "c2l object missing obsm['spatial']"
adata.obsm["spatial"] = c2l[adata.obs_names].obsm["spatial"].copy()

# Optional: do neighbor search within sample if barcodes encode sample suffix
adata.obs["sample_id"] = adata.obs_names.to_series().astype(str).str.split("_", n=1).str[1]

K = 6                 # Visium hex ~6 immediate neighbors
MIN_MZ_NEIGHBORS = 5  # strict: almost fully surrounded (try 4 if too few spots)

is_mz = (adata.obs[CLUSTER_KEY] == MZ_CLUSTER).to_numpy()
core = np.zeros(adata.n_obs, dtype=bool)

for sample, idx in adata.obs.groupby("sample_id", observed=True).indices.items():
    idx = np.asarray(list(idx))
    coords = np.asarray(adata.obsm["spatial"][idx])
    labels = is_mz[idx]

    nn = NearestNeighbors(n_neighbors=min(K + 1, len(idx)), metric="euclidean")
    nn.fit(coords)
    neigh = nn.kneighbors(return_distance=False)[:, 1:]  # drop self

    n_mz = labels[neigh].sum(axis=1)
    core_local = labels & (n_mz >= MIN_MZ_NEIGHBORS)
    core[idx] = core_local

adata.obs["MZ_core"] = core
adata.obs["mz_de_group"] = np.where(core, "MZ_core", "rest")
# important: boundary MZ spots go into rest OR get dropped — drop is cleaner:
keep = core | ~is_mz
ad = adata[keep].copy()
ad.obs["mz_de_group"] = np.where(ad.obs["MZ_core"].to_numpy(), "MZ_core", "rest")
ad.obs["mz_de_group"] = ad.obs["mz_de_group"].astype("category")

print("All MZ spots:", int(is_mz.sum()))
print("Core MZ spots:", int(core.sum()))
print("Dropped boundary MZ:", int(is_mz.sum() - core.sum()))

sc.tl.rank_genes_groups(
    ad,
    groupby="mz_de_group",
    groups=["MZ_core"],
    reference="rest",
    method="wilcoxon",
    pts=True,
    use_raw=False,
)

df_core = sc.get.rank_genes_groups_df(ad, group="MZ_core").rename(
    columns={"names": "gene", "logfoldchanges": "log2FC", "pvals_adj": "padj"}
)
df_core["delta_pct"] = df_core["pct_nz_group"] - df_core["pct_nz_reference"]

strict_core = df_core[
    (df_core["padj"] < 0.05)
    & (df_core["log2FC"] >= 0.75)
    & (df_core["delta_pct"] >= 0.15)
    & (df_core["pct_nz_reference"] <= 0.70)
].sort_values(["log2FC", "delta_pct"], ascending=False)

print(len(strict_core), "strict core-MZ candidates")
display(strict_core.head(30))
df_core.to_csv(OUTDIR / "MZ_core_vs_rest_infected_full.csv", index=False)
strict_core.to_csv(OUTDIR / "MZ_core_vs_rest_infected_STRICT.csv", index=False)

All MZ spots: 242
Core MZ spots: 6
Dropped boundary MZ: 236
0 strict core-MZ candidates


,gene,scores,log2FC,pvals,padj,pct_nz_group,pct_nz_reference,delta_pct


In [2]:
import io
import re
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path
from scipy import sparse

OUT_XLSX = Path("/Users/yashkulkarni/Desktop/GSEAPY/CD8_Klf2_hi_vs_lo_infected_DE_073126.xlsx")
FDR = 0.05
GENE = "Klf2"
Q_LO, Q_HI = 0.25, 0.75

cd8_fine = [
    "CD8-Tcell_Gamma-Delta-CD8", "CD8-Tcell_Gamma-Delta-active",
    "CD8-Tcell_early-active", "CD8-Tcell_effector", "CD8-Tcell_late-active",
    "CD8-Tcell_naive", "CD8-Tcell_proliferating", "CD8-Tcell_terminal", "CD8_NKT",
]


def _to_dense(X):
    return X.toarray() if sparse.issparse(X) else np.asarray(X)


def is_junk_gene(gene: str) -> bool:
    g = str(gene)
    if re.match(r"^Rp[sl]", g, flags=re.I):
        return True
    if re.match(r"^Mrp[sl]", g, flags=re.I):
        return True
    if g.startswith(("mt-", "MT-")):
        return True
    if re.match(r"^Hb[abq]", g, flags=re.I):
        return True
    if g.startswith("Gm") and g[2:].isdigit():
        return True
    if g.endswith("Rik"):
        return True
    return False


# --- infected coarse CD8 ---
ad = adata_sc[
    (adata_sc.obs["timepoint"].astype(str) == "3wk")
    & (adata_sc.obs["coarse_redo"].astype(str) == "CD8-Tcell")
].copy()

if GENE not in ad.var_names:
    raise ValueError(f"{GENE} not in var_names")

klf2 = _to_dense(ad[:, GENE].X).ravel()
ad.obs["Klf2"] = klf2

lo_thr = np.quantile(klf2, Q_LO)
hi_thr = np.quantile(klf2, Q_HI)

ad.obs["Klf2_group"] = "mid"
ad.obs.loc[ad.obs["Klf2"] <= lo_thr, "Klf2_group"] = "Klf2_lo"   # bottom 25%
ad.obs.loc[ad.obs["Klf2"] >= hi_thr, "Klf2_group"] = "Klf2_hi"   # top 25%

print(ad.obs["Klf2_group"].value_counts())
print(f"Klf2 thresholds: lo<={lo_thr:.4f}, hi>={hi_thr:.4f}")

# --- composition sheet ---
sub = ad[ad.obs["Klf2_group"].isin(["Klf2_hi", "Klf2_lo"])].copy()
comp = (
    sub.obs.groupby(["Klf2_group", "celltypes_redo"], observed=False)
    .size()
    .rename("n_cells")
    .reset_index()
)
comp["pct_within_group"] = comp.groupby("Klf2_group")["n_cells"].transform(
    lambda s: 100 * s / s.sum()
)
# also wide summary
ct = pd.crosstab(sub.obs["celltypes_redo"], sub.obs["Klf2_group"])
for col in ["Klf2_hi", "Klf2_lo"]:
    if col not in ct.columns:
        ct[col] = 0
ct["pct_Klf2_hi"] = 100 * ct["Klf2_hi"] / ct["Klf2_hi"].sum()
ct["pct_Klf2_lo"] = 100 * ct["Klf2_lo"] / ct["Klf2_lo"].sum()
ct = ct.reset_index().rename(columns={"celltypes_redo": "fine_celltype"})

summary = pd.DataFrame({
    "metric": [
        "n_infected_coarse_CD8",
        "n_Klf2_hi_top25",
        "n_Klf2_lo_bottom25",
        "Klf2_lo_threshold",
        "Klf2_hi_threshold",
        "Klf2_hi_mean",
        "Klf2_lo_mean",
    ],
    "value": [
        ad.n_obs,
        int((ad.obs["Klf2_group"] == "Klf2_hi").sum()),
        int((ad.obs["Klf2_group"] == "Klf2_lo").sum()),
        lo_thr,
        hi_thr,
        float(ad.obs.loc[ad.obs["Klf2_group"] == "Klf2_hi", "Klf2"].mean()),
        float(ad.obs.loc[ad.obs["Klf2_group"] == "Klf2_lo", "Klf2"].mean()),
    ],
})

# --- DE: Klf2_hi vs Klf2_lo ---
keep = [g for g in sub.var_names if not is_junk_gene(g)]
sub = sub[:, keep].copy()
X = _to_dense(sub.X)
expressed = X.sum(axis=0) > 0
sub = sub[:, expressed].copy()

sc.tl.rank_genes_groups(
    sub,
    groupby="Klf2_group",
    groups=["Klf2_hi"],
    reference="Klf2_lo",
    method="wilcoxon",
    use_raw=False,
)
de = sc.get.rank_genes_groups_df(sub, group="Klf2_hi").rename(columns={
    "names": "gene",
    "logfoldchanges": "log2fc",
    "pvals": "pval",
    "pvals_adj": "padj",
    "scores": "wilcoxon_score",
})

genes = sub.var_names.to_list()
Xh = _to_dense(sub[sub.obs["Klf2_group"] == "Klf2_hi"].X)
Xl = _to_dense(sub[sub.obs["Klf2_group"] == "Klf2_lo"].X)
stats = pd.DataFrame({
    "gene": genes,
    "mean_Klf2_hi": Xh.mean(axis=0),
    "mean_Klf2_lo": Xl.mean(axis=0),
    "pct_Klf2_hi": (Xh > 0).mean(axis=0) * 100,
    "pct_Klf2_lo": (Xl > 0).mean(axis=0) * 100,
})
de = de.merge(stats, on="gene", how="left")
de["significant"] = (de["padj"] < FDR)
de = de.sort_values(["padj", "log2fc"], ascending=[True, False]).reset_index(drop=True)

buf = io.BytesIO()
with pd.ExcelWriter(buf, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="summary", index=False)
    ct.to_excel(writer, sheet_name="fine_type_counts", index=False)
    comp.to_excel(writer, sheet_name="fine_type_long", index=False)
    de.to_excel(writer, sheet_name="DE_Klf2_hi_vs_lo", index=False)
OUT_XLSX.write_bytes(buf.getvalue())
print(f"Wrote {OUT_XLSX}")
print(f"DE genes: {len(de)}, significant: {de['significant'].sum()}")

NameError: name 'adata_sc' is not defined